# 1. Lecture

## 1.1. Implementing the Indexing Pipeline

In [1]:
from utils import index_my_data

In [2]:
# FILE_PATH = "/home/mert/Documents/Mert Gul - CV.pdf"
# INDEX_NAME = "mert-resume"

index_my_data()

1. Data loader is loaded.
2. Data mapper is loaded.
3. Data indexer is loaded.
4. Documents has been retrieved.
5. Documents has been transformed.
	INDEXING SUCCESSFUL - upserted_count: 32

	INDEXING SUCCESSFUL - upserted_count: 10

6. Data has been indexed successfully.


# 1.2. Implementing the Retrieval API

In [3]:
from langserve import RemoteRunnable

Run:

```bash
$ uvicorn server_app:app
```

Make API calls:

In [4]:
rag_chain = RemoteRunnable(url="http://127.0.0.1:8000/rag")

In [20]:
prompt = """\
List the answer to these questions in bullet points:
* What is the latest job position of Mert Gül? 
* What is the latest company Mert Gül had worked?
* List me all the companies Mert Gül had worked to date.
* Which university did Mert Gül graduate?
* Give me the tech stack of Mert Gül.
"""
chain_response = rag_chain.invoke(
    input={
        "input": prompt
    }
)
answer = chain_response["answer"]
print(answer)

- Latest job position: Machine Learning Engineer
- Latest company: Not specified in the provided context
- Companies Mert Gül has worked for: Not specified in the provided context
- University graduated: Middle East Technical University
- Tech stack: Deep learning frameworks, data science, computer vision, NLP libraries, custom LLMs, diffusion models, cloud services for model deployment


In [12]:
import os
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    api_key=os.environ.get("OPENAI_API_KEY")
)

In [21]:
f = open("query_embeddings.txt", 'w')
f.write(
    ','.join(
        list(
            map(
                str,
                embeddings.embed_query(
                    text="Which university did Mert Gul graduate?"
                )
            )
        )
    )
)
f.close()

# 2. Homework Sandbox

Ping my HF Space:

In [2]:
import requests

In [3]:
URL = "https://[USERNAME]-project6-backend.hf.space"

response = requests.get(URL)
response.json()

{'Hello': 'World!'}

In [4]:
from langchain_core.prompts import PromptTemplate

In [25]:
def format_prompt(prompt) -> PromptTemplate:
    # TODO: format the input prompt by using the model specific instruction template
    # TODO: return a langchain PromptTemplate
    PROMPT_TEMPLATE = f"""
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    You are a helpful AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
    {prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
    return PromptTemplate.from_template(
        template=PROMPT_TEMPLATE,
        template_format="f-string"
    )
format_prompt("--mert--")

PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\n    <|begin_of_text|><|start_header_id|>system<|end_header_id|>\n    You are a helpful AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n    --mert--<|eot_id|><|start_header_id|>assistant<|end_header_id|>')

In [ ]:
import transformers
import torch

model_id = "dunzhang/stella_en_1.5B_v5"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)
messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = pipeline(
    messages,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
print(outputs[0]["generated_text"][-1])


In [50]:
pipeline.model.config.hidden_size

1536